# Hotel Review Intelligence - API Validation Notebook
Run FastAPI first: `uvicorn backend.api.main:app --reload`

In [ ]:
import requests, json, pandas as pd
BASE_URL="http://localhost:8000/api/v1"


In [ ]:
health=requests.get(f"{BASE_URL}/health")
print(health.status_code)
print(health.json())

In [ ]:
models=requests.get(f"{BASE_URL}/models")
print(json.dumps(models.json(),indent=2))

## Single Review Test

In [ ]:
review = "The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend. The staff even left a complimentary bottle of wine in our room. Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles. The clawfoot tub looked nice, but the water pressure was practically non-existent. Lovely ambiance and sweet gestures, but the plumbing definitely needs a modern upgrade."
payload = {"review_text": review}
resp = requests.post(f"{BASE_URL}/analyze", json=payload)
data = resp.json()
print(json.dumps(data, indent=2))

## Business Traveler Test

In [ ]:
review="The location is unbeatable for corporate meetings. However the Wi-Fi kept dropping every ten minutes and client calls were impossible."
resp=requests.post(f"{BASE_URL}/analyze",json={"review_text":review})
print(json.dumps(resp.json(),indent=2))

## Family Traveler Test

In [ ]:
review="The kids loved the water park and game room. Check-in took over an hour despite our reservation."
resp=requests.post(f"{BASE_URL}/analyze",json={"review_text":review})
print(json.dumps(resp.json(),indent=2))

## Batch API Test

In [ ]:
payload={"reviews":[{"review_text":"Amazing location and service"},{"review_text":"Dirty room and rude staff"}]}
resp=requests.post(f"{BASE_URL}/analyze-batch",json=payload)
print(json.dumps(resp.json(),indent=2))

## Extract Key Fields Into DataFrame

In [ ]:
reviews=["Amazing hotel and great staff","Dirty room and poor maintenance","Family loved the water park but check-in was chaotic"]
rows=[]
for r in reviews:
    d=requests.post(f"{BASE_URL}/analyze",json={"review_text":r}).json()
    rows.append({
      "sentiment":d["sentiment"]["label"],
      "rating":d["rating"]["prediction"],
      "emotion":d["emotion"]["dominant"],
      "quality":d["quality"]["status"]
    })
pd.DataFrame(rows)

## Aspect Module break-down

In [1]:
from __future__ import annotations  # noqa: F404

import re
from typing import Any

In [2]:
# ==========================================================
# Hotel Aspect Taxonomy (V2)
# ==========================================================

ASPECTS = {

    "room": [
        "room",
        "suite",
        "accommodation",
    ],

    "bed_comfort": [
        "bed",
        "mattress",
        "pillow",
        "blanket",
    ],

    "bathroom": [
        "bathroom",
        "toilet",
        "shower",
        "sink",
        "drain",
    ],

    "sleep_quality": [
        "sleep",
        "slept",
        "rest",
        "awake",
        "insomnia",
    ],

    "staff_service": [
        "staff",
        "service",
        "housekeeping",
        "employee",
        "reception",
        "front desk",
    ],

    "checkin_checkout": [
        "check in",
        "check-in",
        "checkin",
        "checkout",
        "reservation",
        "queue",
        "waited",
        "waiting",
    ],

    "management_resolution": [
        "manager",
        "management",
        "refund",
        "compensation",
        "resolved",
        "resolution",
    ],

    "cleanliness": [
        "clean",
        "dirty",
        "dust",
        "dusty",
        "stain",
        "smell",
        "odor",
        "odour",
    ],

    "maintenance": [
        "broken",
        "repair",
        "damaged",
        "clogged",
        "maintenance",
        "air conditioning",
        "ac unit",
        "air conditioner",
    ],

    "amenities": [
        "pool",
        "gym",
        "spa",
    ],

    "wifi_internet": [
        "wifi",
        "wi-fi",
        "internet",
        "network",
        "connection",
    ],

    "restaurant": [
        "restaurant",
        "dinner",
        "lunch",
        "food",
        "menu",
        "tasting menu",
    ],

    "breakfast": [
        "breakfast",
        "buffet",
        "pastry",
    ],

    "bar_lounge": [
        "bar",
        "cocktail",
        "cocktails",
        "wine",
        "drinks",
    ],

    "location": [
        "location",
        "downtown",
        "city center",
        "city centre",
    ],

    "transport_access": [
        "airport",
        "transit",
        "train",
        "walkable",
        "transport",
    ],

    "view_scenery": [
        "view",
        "ocean",
        "sea",
        "rooftop",
        "scenery",
    ],

    "family_facilities": [
        "family",
        "kids",
        "children",
        "water park",
        "waterpark",
        "game room",
    ],

    "business_facilities": [
        "business",
        "corporate",
        "meeting",
        "meetings",
        "remote work",
        "client call",
        "conference",
    ],

    "value_for_money": [
        "price",
        "cheap",
        "cost",
        "expensive",
        "value",
    ],

    "parking": [
        "parking",
        "garage",
    ],

    "noise": [
        "noise",
        "noisy",
        "loud",
        "hallway",
        "neighbors",
        "neighbours",
        "quiet",
    ],
}

# ==========================================================
# Priority Ranking
# ==========================================================

ASPECT_PRIORITY = {

    "wifi_internet": 10,
    "cleanliness": 10,
    "sleep_quality": 10,

    "staff_service": 9,
    "checkin_checkout": 9,

    "maintenance": 8,
    "bed_comfort": 8,

    "room": 7,
    "value_for_money": 7,
    "location": 7,

    "restaurant": 6,
    "breakfast": 6,
    "bar_lounge": 6,

    "parking": 5,
    "view_scenery": 5,

    "family_facilities": 5,
    "business_facilities": 5,

    "noise": 5,
}

# ==========================================================
# Sentiment Keywords
# ==========================================================

POSITIVE_WORDS = {
    "great",
    "excellent",
    "amazing",
    "stunning",
    "beautiful",
    "clean",
    "comfortable",
    "friendly",
    "helpful",
    "professional",
    "perfect",
    "love",
    "loved",
    "pleasant",
    "kind",
    "spacious",
    "breathtaking",
    "memorable",
    "unbeatable",
}

NEGATIVE_WORDS = {
    "bad",
    "poor",
    "dirty",
    "terrible",
    "awful",
    "rude",
    "slow",
    "broken",
    "clogged",
    "noisy",
    "disappointing",
    "disappointed",
    "frustrating",
    "nightmare",
    "chaos",
    "stressful",
    "uncomfortable",
    "loud",
    "problem",
    "issue",
}

In [3]:
# ==========================================================
# Helpers
# ==========================================================

def split_sentences(text: str) -> list[str]:
    """
    Split text into meaningful sentences.
    """
    parts = re.split(
        r"[.!?;\n]+",
        text,
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


def normalize(text: str) -> str:
    return text.lower().strip()


def calculate_sentiment_score(
    sentence: str,
) -> tuple[str, float]:
    """
    Determine sentiment and confidence.
    """

    text = normalize(sentence)

    positive_hits = sum(
        1
        for word in POSITIVE_WORDS
        if word in text
    )

    negative_hits = sum(
        1
        for word in NEGATIVE_WORDS
        if word in text
    )

    score = positive_hits - negative_hits

    if score > 0:
        confidence = min(
            0.60 + (score * 0.10),
            0.95,
        )

        return (
            "positive",
            round(confidence, 2),
        )

    if score < 0:
        confidence = min(
            0.60 + (abs(score) * 0.10),
            0.95,
        )

        return (
            "negative",
            round(confidence, 2),
        )

    return (
        "neutral",
        0.50,
    )

In [4]:
input_text = "The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend. The staff even left a complimentary bottle of wine in our room. Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles. The clawfoot tub looked nice, but the water pressure was practically non-existent. Lovely ambiance and sweet gestures, but the plumbing definitely needs a modern upgrade."

In [5]:
review_text = re.sub(
        r"\s+",
        " ",
        input_text
    ).strip()

In [7]:
sentences = split_sentences(
    review_text
)
sentences

['The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend',
 'The staff even left a complimentary bottle of wine in our room',
 'Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles',
 'The clawfoot tub looked nice, but the water pressure was practically non-existent',
 'Lovely ambiance and sweet gestures, but the plumbing definitely needs a modern upgrade']

In [13]:
detected_aspects = []

seen_pairs = set()

for sentence in sentences:

    sentence_norm = normalize(
        sentence
    )

    print(f"Processing sentence: {sentence}")

    for (
        aspect,
        keywords,
    ) in ASPECTS.items():

        matched = any(
            keyword in sentence_norm
            for keyword in keywords
        )

        print(f"Checking aspect '{aspect}' with keywords {keywords}: Matched = {matched}")

        if not matched:
            continue

        pair = (
            aspect,
            sentence,
        )

        print(f"Aspect-Sentence pair: {pair}")

        if pair in seen_pairs:
            continue

        seen_pairs.add(pair)

        sentiment, confidence = (
            calculate_sentiment_score(
                sentence
            )
        )

        detected_aspects.append(
            {
                "aspect": aspect,
                "sentiment": sentiment,
                "confidence": confidence,
                "priority":
                    ASPECT_PRIORITY.get(
                        aspect,
                        1,
                    ),
                "evidence": sentence,
            }
        )
        print(f"Detected aspect '{aspect}' with sentiment '{sentiment}' and confidence {confidence}")
    # break



Processing sentence: The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend
Checking aspect 'room' with keywords ['room', 'suite', 'accommodation']: Matched = False
Checking aspect 'bed_comfort' with keywords ['bed', 'mattress', 'pillow', 'blanket']: Matched = False
Checking aspect 'bathroom' with keywords ['bathroom', 'toilet', 'shower', 'sink', 'drain']: Matched = False
Checking aspect 'sleep_quality' with keywords ['sleep', 'slept', 'rest', 'awake', 'insomnia']: Matched = False
Checking aspect 'staff_service' with keywords ['staff', 'service', 'housekeeping', 'employee', 'reception', 'front desk']: Matched = False
Checking aspect 'checkin_checkout' with keywords ['check in', 'check-in', 'checkin', 'checkout', 'reservation', 'queue', 'waited', 'waiting']: Matched = False
Checking aspect 'management_resolution' with keywords ['manager', 'management', 'refund', 'compensation', 'resolved', 'resolution']: Matched = False
Checking aspect 'c

In [ ]:
detected_aspects.sort(
    key=lambda item: (
        -item["priority"],
        -item["confidence"],
    )
)

In [16]:
# ==========================================================
# Main ABSA Node
# ==========================================================

def analyze_aspects(
    review_text: str
) -> dict[str, Any]:
    """
    LangGraph ABSA node.

    Input:
        cleaned_text

    Output:
        aspects[]
    """

    if not review_text:
        return {
            "aspects": []
        }

    sentences = split_sentences(
        review_text
    )

    detected_aspects = []

    seen_pairs = set()

    for sentence in sentences:

        sentence_norm = normalize(
            sentence
        )

        for (
            aspect,
            keywords,
        ) in ASPECTS.items():

            matched = any(
                keyword in sentence_norm
                for keyword in keywords
            )

            if not matched:
                continue

            pair = (
                aspect,
                sentence,
            )

            if pair in seen_pairs:
                continue

            seen_pairs.add(pair)

            sentiment, confidence = (
                calculate_sentiment_score(
                    sentence
                )
            )

            detected_aspects.append(
                {
                    "aspect": aspect,
                    "sentiment": sentiment,
                    "confidence": confidence,
                    "priority":
                        ASPECT_PRIORITY.get(
                            aspect,
                            1,
                        ),
                    "evidence": sentence,
                }
            )

    detected_aspects.sort(
        key=lambda item: (
            -item["priority"],
            -item["confidence"],
        )
    )

    return {
        "aspects": detected_aspects
    }

In [17]:
result = analyze_aspects(
    review_text=review_text
)
result

{'aspects': [{'aspect': 'cleanliness',
   'sentiment': 'neutral',
   'confidence': 0.5,
   'priority': 10,
   'evidence': 'Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles'},
  {'aspect': 'staff_service',
   'sentiment': 'neutral',
   'confidence': 0.5,
   'priority': 9,
   'evidence': 'The staff even left a complimentary bottle of wine in our room'},
  {'aspect': 'room',
   'sentiment': 'neutral',
   'confidence': 0.5,
   'priority': 7,
   'evidence': 'The staff even left a complimentary bottle of wine in our room'},
  {'aspect': 'room',
   'sentiment': 'neutral',
   'confidence': 0.5,
   'priority': 7,
   'evidence': 'Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles'},
  {'aspect': 'bar_lounge',
   'sentiment': 'neutral',
   'confidence': 0.5,
   'priority': 6,
   'evidence': 'The staff even left a complimentary bottle of wine in our room'},
  {'aspect': 'bathroom',
   'sentiment': 'neutral',
   'confidence

In [ ]:
# ==========================================================
# Main ABSA Node
# ==========================================================

def analyze_aspects(
    state: ReviewState,
) -> dict[str, Any]:
    """
    LangGraph ABSA node.

    Input:
        cleaned_text

    Output:
        aspects[]
    """

    review_text = state.get(
        "cleaned_text",
        "",
    )

    if not review_text:
        return {
            "aspects": []
        }

    sentences = split_sentences(
        review_text
    )

    detected_aspects = []

    seen_pairs = set()

    for sentence in sentences:

        sentence_norm = normalize(
            sentence
        )

        for (
            aspect,
            keywords,
        ) in ASPECTS.items():

            matched = any(
                keyword in sentence_norm
                for keyword in keywords
            )

            if not matched:
                continue

            pair = (
                aspect,
                sentence,
            )

            if pair in seen_pairs:
                continue

            seen_pairs.add(pair)

            sentiment, confidence = (
                calculate_sentiment_score(
                    sentence
                )
            )

            detected_aspects.append(
                {
                    "aspect": aspect,
                    "sentiment": sentiment,
                    "confidence": confidence,
                    "priority":
                        ASPECT_PRIORITY.get(
                            aspect,
                            1,
                        ),
                    "evidence": sentence,
                }
            )

    detected_aspects.sort(
        key=lambda item: (
            -item["priority"],
            -item["confidence"],
        )
    )

    return {
        "aspects": detected_aspects
    }